In [1]:
import json
import os
import kagglehub
import importlib
import torch
import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
import pandas as pd
import numpy as np
import gc
from src.utils import load_indices_from_jsonl
from IPython.display import JSON
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from functools import partial
from src.jsonl_dataset import JsonLDataset
from transformers import AutoTokenizer, TrainingArguments, EvalPrediction
from adapters import AutoAdapterModel, AdapterTrainer
from sklearn.metrics import f1_score, precision_score, recall_score
from adapters.composition import Parallel 

### Setup

In [2]:
path = kagglehub.dataset_download("Cornell-University/arxiv/versions/272")
ds_path = os.path.join(path, 'arxiv-metadata-oai-snapshot.json')
master_ds = JsonLDataset(ds_path)
augmented_ds = JsonLDataset("resources/augmented_index.jsonl")

Indexing dataset at C:\Users\kerem\.cache\kagglehub\datasets\Cornell-University\arxiv\versions\272\arxiv-metadata-oai-snapshot.json... (this may take a minute)
Indexed 2951540 entries.
Indexing dataset at resources/augmented_index.jsonl... (this may take a minute)
Indexed 2951540 entries.


In [19]:
from src.multiadapter_parent_predictor import MultiAdapterParentPredictor
from src import subcategory_predictor

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer
def create_mlb(target_classes):
    mlb = MultiLabelBinarizer(classes=target_classes)
    mlb.fit([target_classes])
    return mlb

In [7]:
def build_adapter_configs():
    TARGET_SUB_CATEGORIES = json.load(open("resources/target_sub_classes.json", "r"))
    result = []
    for sub_cat_name, classes in TARGET_SUB_CATEGORIES.items():
        result.append({
            "category": sub_cat_name,
            "name": f"{sub_cat_name}_categories_adapter",
            "path": f"./resources/{sub_cat_name}_categories_adapter",
            "mbl": create_mlb(classes)
        })
    return result

In [189]:
importlib.reload(subcategory_predictor)

<module 'src.subcategory_predictor' from 'C:\\projects\\personal\\python-notebooks\\interview_irisai\\src\\subcategory_predictor.py'>

In [190]:
subc_predictor = subcategory_predictor.SubcategoryPredictor(
    adapter_configs=build_adapter_configs()
)

Initializing Multi-Adapter Pipeline on cuda...
  -> Loading Physics_categories_adapter...
  -> Loading Mathematics_categories_adapter...
  -> Loading Computer Science_categories_adapter...
  -> Loading Quantitative Biology_categories_adapter...
  -> Loading Statistics_categories_adapter...
  -> Loading Quantitative Finance_categories_adapter...
  -> Loading Economics_categories_adapter...
  -> Loading Electrical Engineering and Systems Science_categories_adapter...


In [209]:
res2 = subc_predictor.predict([random.choice(master_ds)], category="Mathematics", threshold=0.8)

In [208]:
sample = random.choice(master_ds)
res = subc_predictor.predict([sample], category="Mathematics", threshold=0.8)

print(f"Predicted: {res[0]["labels"]}")
print("-" * 50)
print(f"Actual: {sample["categories"]}")
print(json.dumps(sample, indent=4))

Predicted: ['math.CO']
--------------------------------------------------
Actual: math.CO
{
    "id": "1404.1708",
    "submitter": "Christian Stump",
    "authors": "Christian Stump",
    "title": "On a new collection of words in the Catalan family",
    "comments": "5 pages, v2: title changed + fixed typos; final version",
    "journal-ref": "Journal of Integer Sequences, Vol. 17 (2014), Article 14.7.1",
    "doi": null,
    "report-no": null,
    "categories": "math.CO",
    "license": "http://arxiv.org/licenses/nonexclusive-distrib/1.0/",
    "abstract": "  In this note, we provide a bijection between a new collection of words on\nnonnegative integers of length n and Dyck paths of length 2n-2, thus proving\nthat this collection belongs to the Catalan family. The surprising key step in\nthis bijection is the zeta map which is an important map in the study of\nq,t-Catalan numbers. Finally we discuss an alternative approach to this new\ncollection of words using two statistics on plan

### Test

In [44]:
def build_input(items, tokenizer):
    texts = [
        f"{item['title']}{tokenizer.sep_token}{item.get('abstract', '')}" 
        for item in items
    ]

    # 2. Tokenization
    return tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=512, 
        return_tensors="pt"
    ).to('cuda')

In [170]:
adapter = build_adapter_configs()[0]
model = AutoAdapterModel.from_pretrained("allenai/specter2_base")
model.load_adapter(adapter["path"], load_as=adapter["name"])
model.set_active_adapters(adapter["name"])
model.to('cuda')
tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")

There are adapters available but none are activated for the forward pass.


In [175]:
with torch.no_grad():
    inp = build_input([master_ds[2]] , tokenizer)
    resp = model(**inp)

In [176]:
resp.logits

tensor([[-2.6802, -0.6276, -6.1964, -6.9250, -5.9620, -5.1285, -1.1566, -5.7073,
         -2.7776, -6.8361, -4.4790, -6.3728, -3.6546, -5.9749, -5.9746, -2.4355,
         -6.6446, -0.1589, -6.8651, -9.8021, -5.4694, -8.7431, -7.1209, -7.5245,
         -6.4583, -8.8785, -5.2850, -5.6960, -5.8092, -9.1870, -7.8208, -8.5552,
         -5.7380, -1.6355, -0.8568, -4.8679,  0.8854, -3.3662, -4.0113, -3.2718,
         -4.7612, -4.1203, -5.4409, -7.3773, -4.9245, -7.1301, -7.0109, -5.5960,
         -7.5645, -7.5072, -6.3459, -6.3881, -6.9156]], device='cuda:0')